In [1]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


# Practice 4 — Challenge B: regression tree (R3)

**Scope:** P4.3, the rules of P4.4 (from the storage tree) and the Challenge B part of P4.5.

> **Team code style (delete this cell before submitting).** The team must be able to defend every line.
>
> - Prefer tools seen in class or named in the statement. If you cannot explain a line in one sentence, rewrite it.
> - Plain `for` loops and `if`s, one step per line, clear variable names. No lambdas, classes or long chains.
> - Same results, simpler code. After simplifying, restart, run all and check that the numbers did not change.
> - Interpretation goes in Markdown cells, in English. Check your notebook with `revisor_notebook.py`.
> - Every number written in a Markdown cell must appear in a printed output of the notebook.
>
> **Shared names, so that the notebooks can be joined:** `dev` (development table), `X`, `y` (cyber target), `z` (storage target), `latency_ns`, `NOMINAL`, `NUMERIC`, `FEATURES`, `make_preprocessor()`, `folds` (list of `(train_idx, val_idx)`, also passed as `cv=folds`), `CLASS_NAMES` (cyber).

## Setup from R1 and R4

Paste here the cells published by R1 and R4. They are **not** pasted into the final notebook.

In [2]:
# TODO: paste the data, preprocessing, folds and metrics cells
import time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Data (R1): the nine source predictors; metadata and iops never enter X
dev = pd.read_csv("../../datos/storage/storage_development.csv")
NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
FEATURES = NOMINAL + NUMERIC
X = dev[FEATURES]
latency_ns = dev["latency_ns"].to_numpy()
z = np.log10(latency_ns)

# Preprocessing for trees
def make_preprocessor():
    return ColumnTransformer([
        # One 0/1 column per level, no level dropped; unseen level -> all zeros
        ("nominal", OneHotEncoder(handle_unknown="ignore", sparse_output=False), NOMINAL),
        # numeric values kept as they are, with no scaling
        ("numeric", "passthrough", NUMERIC),
    ])

# Canonical folds (R4): read from the course file, never rebuilt
fold_table = pd.read_csv("../../folds/storage_folds.csv")
assert np.array_equal(fold_table["row_id"], dev["row_id"]), "rows are not aligned"
fold_id = fold_table["fold_id"].to_numpy()
folds = []
for f in range(5):
    train_idx = np.where(fold_id != f)[0]
    val_idx = np.where(fold_id == f)[0]
    folds.append((train_idx, val_idx))

# Five per-fold values of one row of comparativa.csv
def table_scores(milestone, challenge, family):
    table = pd.read_csv("../../comparativa.csv")
    is_row = ((table["milestone"] == milestone) & (table["challenge"] == challenge) & (table["family"] == family))
    text = table.loc[is_row, "primary_metric_per_fold"].iloc[0]
    return np.array(text.split(), dtype = float)

print("X:", X.shape, "| z:", z.shape)
print("Transformed columns:", make_preprocessor().fit(X).get_feature_names_out().shape[0])
for f in range(5):
    print("fold", f, "| training rows:", len(folds[f][0]), "| validation rows:", len(folds[f][1]))

print("P2 KNN:", table_scores("P2", "storage", "KNN"))
print("P3 LinearRegression:", table_scores("P3", "storage", "LinearRegression"))

X: (12190, 9) | z: (12190,)
Transformed columns: 17
fold 0 | training rows: 9750 | validation rows: 2440
fold 1 | training rows: 9750 | validation rows: 2440
fold 2 | training rows: 9750 | validation rows: 2440
fold 3 | training rows: 9755 | validation rows: 2435
fold 4 | training rows: 9755 | validation rows: 2435
P2 KNN: [0.1816 0.187  0.1778 0.1775 0.1762]
P3 LinearRegression: [0.2047 0.1937 0.1984 0.1953 0.1987]


# P4.3 — Tree regression

In [3]:
# TODO: imports for this part
import time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import export_text, plot_tree

In [5]:
# TODO: declare the grid values for this task
def make_tree():
    # A new, unfitted pipeline every time: each fold gets a fresh copy,
    # so nothing learned in one fold leaks into another
    return Pipeline([
        ("preprocessing", make_preprocessor()),  # one-hot nominal columns, pass numeric ones
        ("tree", DecisionTreeRegressor(random_state=PROJECT_RANDOM_STATE)),  # learns to predict z
    ])

# Check 1: how many rows each configuration has
rows_per_config = dev.groupby(FEATURES).size()
print("Number of configurations:", len(rows_per_config))
print("Configurations by number of rows:")
print(rows_per_config.value_counts())

# Fold number of each row, taken from the shared folds
fold_of_row = np.zeros(len(dev), dtype=int)
for f in range(5):
    val_idx = folds[f][1]
    fold_of_row[val_idx] = f

# Check 2: in how many different folds each configuration appears (we expect always 1)
dev_with_fold = dev.copy()
dev_with_fold["fold"] = fold_of_row
folds_per_config = dev_with_fold.groupby(FEATURES)["fold"].nunique()
print("Configurations by number of different folds:")
print(folds_per_config.value_counts())

# Declared grid: names start with "tree__" because the tree is the "tree" step of the pipeline
param_grid = {
    "tree__max_depth": [4, 6, 8, 10, 12, None],      # None = no depth limit
    "tree__min_samples_leaf": [5, 10, 20, 40],       # multiples of 5 replicates
    "tree__ccp_alpha": [0.0, 0.00001, 0.0001, 0.001],# pruning, in units of squared error of z
}

# Numbers used in the Markdown cells
for leaf in param_grid["tree__min_samples_leaf"]:
    print("min_samples_leaf =", leaf, "-> at least", leaf // 5, "configurations per leaf")
print("Variance of z:", round(z.var(), 4))

n_combinations = 1
for name in param_grid:
    n_combinations = n_combinations * len(param_grid[name])
print("Combinations:", n_combinations, "| fits:", n_combinations*len(folds))

Number of configurations: 2438
Configurations by number of rows:
5    2438
Name: count, dtype: int64
Configurations by number of different folds:
fold
1    2438
Name: count, dtype: int64
min_samples_leaf = 5 -> at least 1 configurations per leaf
min_samples_leaf = 10 -> at least 2 configurations per leaf
min_samples_leaf = 20 -> at least 4 configurations per leaf
min_samples_leaf = 40 -> at least 8 configurations per leaf
Variance of z: 0.7056
Combinations: 96 | fits: 480


Each configuration appears 5 times with identical predictors, so a tree can never split its replicates: every leaf already holds >=5 rows. Values below 5 would give the same tree. We use multiples of 5 (= at least 1, 2, 4, 8 configurations per leaf).

**Declared grid and why.** TODO

In [6]:
# TODO: GridSearchCV(pipe, param_grid, cv=folds, scoring='neg_mean_absolute_error', return_train_score=True)
search = GridSearchCV(
    make_tree(),
    param_grid,
    cv=folds,                                # the shared canonical folds
    scoring="neg_mean_absolute_error",       # MAE(z), negative because sklearn maximises
    return_train_score=True,                 # also keep training error, to see overfitting
)

start = time.time()
search.fit(X, z)
search_seconds = time.time() - start
print("Grid search time (s):", round(search_seconds, 1))

Grid search time (s): 23.4


In [7]:
# TODO: the selected configuration
best = search.best_params_
best_index = search.best_index_

print("Selected configuration:")
for name in best:
    print(" ", name, "=", best[name])

print("Mean validation MAE(z):", round(-search.best_score_, 4))
print("Mean training MAE(z):", round(-search.cv_results_["mean_train_score"][best_index], 4))

# If the selected value is at an edge, the grid may be too short on that side
for name in param_grid:
    values = param_grid[name]
    if best[name] == values[0] or best[name] == values[-1]:
        print("At the edge of the grid:", name, "=", best[name])

Selected configuration:
  tree__ccp_alpha = 0.0
  tree__max_depth = 12
  tree__min_samples_leaf = 5
Mean validation MAE(z): 0.1306
Mean training MAE(z): 0.0324
At the edge of the grid: tree__min_samples_leaf = 5
At the edge of the grid: tree__ccp_alpha = 0.0


In [ ]:
# TODO: MAE(z) for each of the five folds, its mean and spread, RMSE(z), R2(z)
# and one MAE in nanoseconds computed after 10 ** z_hat

**Interpretation.** TODO

**What would have gone wrong had the classification hyperparameters been reused.** TODO: sample size, the structure of the rows, the impurity minimised, and the units of ccp_alpha.

# P4.4 — Rules

In [ ]:
# TODO: say which fitted tree the rules come from (the selected one, or a shallower tree for reading only)

In [ ]:
# TODO: the full rule set of two named leaves, each with its node, predicted value
# and number of training rows (required figure)

**One rule read as an operations statement.** TODO: on the physical scale (latency = 10 ** z_hat ns).

# P4.5 — Challenge B against Practice 3 and Practice 2

In [ ]:
# TODO: the paired per-fold differences against the Practice 3 linear regression and the Practice 2 KNN

### The six fixed comparison questions

Answer each one for both comparisons: the tree against the Practice 3 linear regression and against the Practice 2 KNN.

1. **Size and consistency of the change.** Using the five paired differences (descriptive only). TODO
2. **Route and representation.** Pipeline comparison or isolated family comparison, and what the evidence licenses. TODO
3. **What can the new family express** that the previous one could not? TODO
4. **What did it cost** in fitting time, tuning effort and interpretability? TODO
5. **Did the ordering of the two challenges' difficulty change**, and why? TODO
6. **Would you ship the new model?** If not, what would change your mind? TODO